# Revision reruns (analytics-4589312): corrected benchmark, ablation, seeds, statistics

Run the cells **in order**. Each cell saves its results to `/content/rerun_results/`, so if Colab disconnects you can re-run from Cell 2 and the benchmark resumes where it stopped.

| Cell | What it does | Reviewer items |
|---|---|---|
| 1 | Install libraries | |
| 2 | Settings (seeds, features, file path) | |
| 3 | Row-level fixes on the full file, then chronological 70 / 10 / 20 split per class, then cleaning fitted on **training data only** | R1-6, R2-2, R3-2 |
| 4 | Feature configurations: paper's 9, 9 + Memory Commit Limit, train-only reselection | R1-4, R1-5, R2-1 |
| 5 | Model definitions (Table 9 settings, validation-based tuning and early stopping) | R1-6, R1-7 |
| 6 | **Main run**: 8 models × 3 feature sets × 5 seeds (longest cell) | R1-7, R2-7, R3-4 |
| 7 | Summary table, mean ± SD | R1-7, R3-4 |
| 8 | FT-Transformer vs RF / XGBoost / LightGBM: McNemar + bootstrap CI | R1-7, R3-4 |
| 9 | Confusion matrix + per-class report (replaces Figure 9 / Table 12 source) | R1-1, R2-7 |
| 10 | Capture-identity test: does a feature separate capture segments of the **same** class? | R1-4, R2-1 |
| 11 | Held-out capture segment test (closest to independent sessions in this file) | R1-3, R2-3, R3-1 |
| 12 | Windows 7 compatibility check (only if you upload that file) | R1-3, R3-1 |
| 13 | Zip and download all results | |

Before Cell 1: **Runtime → Change runtime type → T4 GPU**, then upload `Train_Test_Windows_10.csv` in the Files panel.

In [ ]:
# Cell 1: install (run once per Colab session)
!pip -q install pytorch-tabnet==4.1.0 rtdl_revisiting_models==0.0.2
import sklearn, torch, xgboost, lightgbm
print('sklearn', sklearn.__version__, '| torch', torch.__version__, '| xgboost', xgboost.__version__,
      '| lightgbm', lightgbm.__version__, '| GPU:', torch.cuda.is_available())

In [ ]:
# Cell 2: settings. Fixed BEFORE looking at any new results; do not change them afterwards.
import glob, os, json, time, copy, random, pickle, warnings
import numpy as np, pandas as pd, torch
warnings.filterwarnings('ignore')

_found = (glob.glob('/content/**/Train_Test_Windows_10*.csv', recursive=True)
          + glob.glob('/Train_Test_Windows_10*.csv'))
DATA_PATH = os.environ.get('W10_PATH') or (_found[0] if _found else None)
assert DATA_PATH, 'Upload Train_Test_Windows_10.csv in the Colab Files panel first.'
OUT = os.environ.get('RERUN_OUT', '/content/rerun_results'); os.makedirs(OUT, exist_ok=True)
print('Data file:', DATA_PATH, '| results folder:', OUT)

SEEDS = [42, 43, 44, 45, 46]            # prespecified seeds
TRAIN_FRAC, VAL_FRAC, GAP_FRAC = 0.70, 0.10, 0.0   # per class, in record order: 70% train | 10% val | 20% test (agreed with the professors)
NZV_THR, CAP_Q, CORR_THR = 0.95, 0.99, 0.95
FT_EPOCHS, TABNET_MAX_EPOCHS, TABNET_PATIENCE, BATCH = 50, 100, 15, 1024   # Table 9 values

FEATURES_9 = [
    'Memory Cache Bytes Peak',
    'Memory System Driver Resident Bytes',
    'Memory Pool Paged Resident Bytes',
    'LogicalDisk(_Total) pct_ Free Space',
    'Network_I(Intel R _82574L_GNC) Packets Sent Unicast sec',
    'Network_I(Intel R _82574L_GNC) Packets Sent sec',
    'Network_I(Intel R _82574L_GNC) Bytes Sent sec',
    'Process_IO Read_Operations_sec',
    'Process_IO_Write Operations_sec',
]
MCL = 'Memory Commit Limit'
FINGERPRINT_PATTERNS = ['TCP_APS', 'Standby Cache Lifetime', 'Free System Page Table Entries']  # the 3 removed by hand in the paper
F_PERCENTILE = 60                        # paper's F-statistic cutoff, now applied to training data only

MODELS = ['LR', 'DT', 'RF', 'XGB', 'LGBM', 'MLP', 'TabNet', 'FT']
CONFIGS = ['9', '10', 'reselect']
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

if os.environ.get('SMOKE') == '1':       # quick test mode, never set in Colab
    SEEDS, FT_EPOCHS, TABNET_MAX_EPOCHS = [42, 43], 2, 3

def norm(s):   # compare column names ignoring spaces, underscores and case
    return ''.join(ch for ch in str(s).lower() if ch.isalnum())

def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(s)

json.dump(dict(SEEDS=SEEDS, TRAIN_FRAC=TRAIN_FRAC, VAL_FRAC=VAL_FRAC, GAP_FRAC=GAP_FRAC, NZV_THR=NZV_THR, CAP_Q=CAP_Q,
               CORR_THR=CORR_THR, FT_EPOCHS=FT_EPOCHS, TABNET_MAX_EPOCHS=TABNET_MAX_EPOCHS,
               TABNET_PATIENCE=TABNET_PATIENCE, BATCH=BATCH, FEATURES_9=FEATURES_9, MCL=MCL,
               F_PERCENTILE=F_PERCENTILE, DEVICE=DEVICE), open(f'{OUT}/settings.json', 'w'), indent=2)

In [ ]:
# Cell 3: load, clean, split. Every learned statistic (variance filter, caps, correlation) uses TRAINING rows only.
from sklearn.preprocessing import LabelEncoder
acct = []
def log(step, rows, cols, note=''):
    acct.append(dict(step=step, rows=rows, columns=cols, note=note))

raw = pd.read_csv(DATA_PATH, low_memory=False)
log('Raw CSV loaded', len(raw), raw.shape[1])
df = raw.copy()
FEATS_ALL = [c for c in df.columns if c not in ('label', 'type')]
empty_before = int(df[FEATS_ALL].isna().sum().sum())
for c in FEATS_ALL:
    df[c] = pd.to_numeric(df[c], errors='coerce')
n_bad = int(df[FEATS_ALL].isna().sum().sum()) - empty_before
log('Text columns converted to numeric', len(df), df.shape[1],
    f'{n_bad} unparseable cells -> NaN ({empty_before} empty cells before conversion)')

df['type'] = df['type'].astype(str).str.strip().str.lower()
df = df[df['type'] != 'mitm'].copy()          # original row index kept = record order
log('MITM removed', len(df), df.shape[1], f'{len(raw) - len(df)} rows')
n_dup = int(df.duplicated().sum())
if n_dup: df = df.drop_duplicates()
log('Duplicate rows', len(df), df.shape[1], f'{n_dup} duplicates removed')
n_neg = int((df[FEATS_ALL] < 0).any(axis=1).sum())
log('Negative-value rows checked', len(df), df.shape[1], f'{n_neg} rows with negatives (kept)')
df[FEATS_ALL] = df[FEATS_ALL].fillna(0)      # constant fill (0), nothing learned from the data, so no test information
log('Targets separated (label, type)', len(df), len(FEATS_ALL), 'NaN filled with constant 0')

# Per-class chronological split
def chrono_split(idx):
    n = len(idx); ntr = int(TRAIN_FRAC * n); nva = int(round(VAL_FRAC * n)); gap = int(round(GAP_FRAC * n))
    return idx[:ntr], idx[ntr + gap: ntr + gap + nva], idx[ntr + 2 * gap + nva:]   # test = the rest (about 20%)

parts, sizes = {'train': [], 'val': [], 'test': []}, {}
for c in sorted(df['type'].unique()):
    idx = np.sort(df.index[df['type'] == c].values)
    tr, va, te = chrono_split(idx)
    parts['train'] += list(tr); parts['val'] += list(va); parts['test'] += list(te)
    sizes[c] = dict(train=len(tr), val=len(va), test=len(te), gap_dropped=len(idx) - len(tr) - len(va) - len(te))
TR, VA, TE = (np.array(sorted(parts[k])) for k in ('train', 'val', 'test'))

# Train-only cleaning
train = df.loc[TR, FEATS_ALL]
NZV = [c for c in FEATS_ALL if train[c].value_counts(normalize=True).iloc[0] > NZV_THR]
KEEP = [c for c in FEATS_ALL if c not in NZV]
log('Near-zero-variance removed (fit on train)', len(TR), len(KEEP), f'{len(NZV)} columns')
CAPS = df.loc[TR, KEEP].quantile(CAP_Q)
X = df[KEEP].clip(upper=CAPS, axis=1)        # caps from train, applied to train / val / test
log('99th-percentile caps (fit on train, applied to all)', len(TR), len(KEEP), 'no rows removed')
corr = X.loc[TR].corr().abs()
upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
CORR_DROP = [c for c in upper.columns if (upper[c] > CORR_THR).any()]
ELIGIBLE = [c for c in KEEP if c not in CORR_DROP]
log('Correlated columns removed (fit on train)', len(TR), len(ELIGIBLE), f'{len(CORR_DROP)} columns, |r|>{CORR_THR}')

le = LabelEncoder().fit(df['type'])
CLASSES = list(le.classes_)
y = pd.Series(le.transform(df['type']), index=df.index)
y_tr, y_va, y_te = y.loc[TR].values, y.loc[VA].values, y.loc[TE].values

acct = pd.DataFrame(acct); split_tab = pd.DataFrame(sizes).T
print(acct.to_string(index=False)); print('\nPer-class split sizes:\n', split_tab.to_string())
print(f'\nTotals: train {len(TR)}, val {len(VA)}, test {len(TE)} | classes {CLASSES}')
print(f'Eligible features after train-only cleaning: {len(ELIGIBLE)}')
acct.to_csv(f'{OUT}/data_accounting.csv', index=False); split_tab.to_csv(f'{OUT}/split_sizes.csv')
json.dump(dict(train=TR.tolist(), val=VA.tolist(), test=TE.tolist(), classes=CLASSES, nzv=NZV,
               corr_drop=CORR_DROP, eligible=ELIGIBLE), open(f'{OUT}/split_and_cleaning.json', 'w'))

In [ ]:
# Cell 4: feature configurations
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import f_classif

missing = [f for f in FEATURES_9 + [MCL] if f not in X.columns]
assert not missing, f'Not found after cleaning (check names / NZV filter): {missing}'
print('Paper features dropped by the train-only correlation filter:',
      [f for f in FEATURES_9 + [MCL] if f in CORR_DROP] or 'none',
      '\n(They are kept in configs "9" and "10" because those sets are pre-specified.)')

# Train-only rerun of the documented selection procedure (paper Section 4.4):
# RF importance -> top 20 -> drop the 3 hand-flagged fingerprint features -> drop F >= 60th percentile -> drop MCL
rf_sel = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1).fit(X.loc[TR, ELIGIBLE], y_tr)
imp = pd.Series(rf_sel.feature_importances_, index=ELIGIBLE).sort_values(ascending=False)
top20 = list(imp.index[:20])
fp = [f for f in top20 if any(norm(p) in norm(f) for p in FINGERPRINT_PATTERNS)]
cand = [f for f in top20 if f not in fp]
Fv, _ = f_classif(X.loc[TR, cand], y_tr)
Fs = pd.Series(np.nan_to_num(Fv, nan=0.0), index=cand).sort_values(ascending=False)
F_CUT = float(np.percentile(Fs.values, F_PERCENTILE))
f_drop = list(Fs[Fs >= F_CUT].index)
RESELECT = [f for f in cand if f not in f_drop and f != MCL]

FEATURE_SETS = {'9': FEATURES_9, '10': FEATURES_9 + [MCL], 'reselect': RESELECT}
sel_report = pd.DataFrame({'rf_importance': imp.loc[top20], 'F_train': Fs.reindex(top20)})
sel_report['decision'] = ['fingerprint (hand-flagged)' if f in fp else f'F >= {F_CUT:.1f}' if f in f_drop
                          else 'MCL (manual rule)' if f == MCL else 'kept' for f in top20]
print(sel_report.to_string())
print(f'\nTrain-only reselection kept {len(RESELECT)} features:', RESELECT)
overlap = sorted(set(RESELECT) & set(FEATURES_9))
print(f'Overlap with the paper\'s 9: {len(overlap)} ->', overlap)
sel_report.to_csv(f'{OUT}/reselection_report.csv')
json.dump(FEATURE_SETS, open(f'{OUT}/feature_sets.json', 'w'), indent=2)

In [ ]:
# Cell 5: models. Settings from Table 9; small grids tuned on the VALIDATION set (macro-F1); test set never used here.
import torch.nn.functional as Fnn
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, f1_score
from xgboost import XGBClassifier
import lightgbm as lgb
from pytorch_tabnet.tab_model import TabNetClassifier
from rtdl_revisiting_models import FTTransformer

def mf1(a, b): return f1_score(a, b, average='macro')

def size_kb(obj):
    try: return len(pickle.dumps(obj)) / 1024
    except Exception: return np.nan

def grid_fit(make, grid, Xtr, ytr, Xva, yva, fit_kw=None):
    best = None
    for params in grid:
        m = make(**params)
        m.fit(Xtr, ytr, **(fit_kw(params) if fit_kw else {}))
        s = mf1(yva, m.predict(Xva))
        if best is None or s > best[0]: best = (s, params, m)
    return best[2], best[1]

def torch_predict(model, Xn):
    model.eval(); out = []
    with torch.no_grad():
        for i in range(0, len(Xn), 4096):
            out.append(model(torch.tensor(Xn[i:i + 4096], dtype=torch.float32, device=DEVICE), None).argmax(1).cpu().numpy())
    return np.concatenate(out)

def train_ft(Xtr, ytr, Xva, yva, seed):
    set_seed(seed)
    model = FTTransformer(n_cont_features=Xtr.shape[1], cat_cardinalities=[], d_out=len(CLASSES),
                          **FTTransformer.get_default_kwargs()).to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-5)
    xt = torch.tensor(Xtr, dtype=torch.float32, device=DEVICE); yt = torch.tensor(ytr, dtype=torch.long, device=DEVICE)
    g = torch.Generator().manual_seed(seed); best = (-1, None, 0)
    for ep in range(FT_EPOCHS):
        model.train()
        for b in torch.randperm(len(xt), generator=g).split(BATCH):
            b = b.to(DEVICE); opt.zero_grad()
            Fnn.cross_entropy(model(xt[b], None), yt[b]).backward(); opt.step()
        s = mf1(yva, torch_predict(model, Xva))
        if s > best[0]: best = (s, copy.deepcopy(model.state_dict()), ep + 1)
    model.load_state_dict(best[1])
    return model, {'best_epoch': best[2]}

def run_model(name, Xtr, ytr, Xva, yva, seed):
    """Returns (fitted model, chosen params, predict function)."""
    if name == 'LR':
        m, p = grid_fit(lambda **k: LogisticRegression(max_iter=1000, **k), [dict(C=c) for c in (0.1, 1, 10)], Xtr, ytr, Xva, yva)
    elif name == 'DT':
        m, p = grid_fit(lambda **k: DecisionTreeClassifier(random_state=seed, **k),
                        [dict(max_depth=d, min_samples_leaf=l) for d in (None, 10, 20) for l in (1, 5)], Xtr, ytr, Xva, yva)
    elif name == 'RF':
        m, p = grid_fit(lambda **k: RandomForestClassifier(n_estimators=100, random_state=seed, n_jobs=-1, **k),
                        [dict(max_depth=d, min_samples_leaf=l) for d in (None, 20) for l in (1, 2)], Xtr, ytr, Xva, yva)
    elif name == 'XGB':
        m, p = grid_fit(lambda **k: XGBClassifier(n_estimators=1000, learning_rate=0.05, subsample=0.9, colsample_bytree=0.9,
                                                  tree_method='hist', early_stopping_rounds=50, eval_metric='mlogloss',
                                                  random_state=seed, n_jobs=-1, **k),
                        [dict(max_depth=d) for d in (4, 6, 8)], Xtr, ytr, Xva, yva,
                        fit_kw=lambda _: dict(eval_set=[(Xva, yva)], verbose=False))
        p = dict(p, best_iteration=int(m.best_iteration))
    elif name == 'LGBM':
        m, p = grid_fit(lambda **k: lgb.LGBMClassifier(n_estimators=2000, learning_rate=0.05, random_state=seed, verbose=-1, n_jobs=-1, **k),
                        [dict(num_leaves=n) for n in (15, 31, 63)], Xtr, ytr, Xva, yva,
                        fit_kw=lambda _: dict(eval_set=[(Xva, yva)], callbacks=[lgb.early_stopping(50, verbose=False)]))
        p = dict(p, best_iteration=int(m.best_iteration_))
    elif name == 'MLP':   # (64, 32) ReLU, 500 iterations, early stopping on a split of the TRAINING data (Table 9)
        m, p = grid_fit(lambda **k: MLPClassifier(hidden_layer_sizes=(64, 32), activation='relu', max_iter=500,
                                                  early_stopping=True, random_state=seed, **k),
                        [dict(alpha=a) for a in (1e-4, 1e-3)], Xtr, ytr, Xva, yva)
    elif name == 'TabNet':  # early stopping now monitors the VALIDATION set (was the test set)
        set_seed(seed)
        m = TabNetClassifier(seed=seed, verbose=0, device_name=DEVICE)
        m.fit(Xtr.astype(np.float32), ytr, eval_set=[(Xva.astype(np.float32), yva)], eval_name=['val'],
              eval_metric=['logloss'], max_epochs=TABNET_MAX_EPOCHS, patience=TABNET_PATIENCE,
              batch_size=BATCH, virtual_batch_size=128)
        p = {'best_epoch': int(m.best_epoch)}
        return m, p, lambda Z: m.predict(Z.astype(np.float32))
    elif name == 'FT':      # 50 epochs as in Table 9; the epoch with the best VALIDATION macro-F1 is kept
        m, p = train_ft(Xtr, ytr, Xva, yva, seed)
        return m, p, lambda Z: torch_predict(m, Z)
    else:
        raise ValueError(name)
    return m, p, m.predict

def n_params(m):
    if isinstance(m, torch.nn.Module): return sum(t.numel() for t in m.parameters())
    if hasattr(m, 'network'): return sum(t.numel() for t in m.network.parameters())
    return np.nan
print('Models ready on', DEVICE)

In [ ]:
# Cell 6: MAIN RUN. Saves after every model, so re-running this cell resumes where it stopped.
from sklearn.preprocessing import StandardScaler
RES_CSV, PRED_NPZ = f'{OUT}/benchmark_results.csv', f'{OUT}/test_predictions.npz'
res = pd.read_csv(RES_CSV).to_dict('records') if os.path.exists(RES_CSV) else []
preds = dict(np.load(PRED_NPZ)) if os.path.exists(PRED_NPZ) else {}
np.save(f'{OUT}/y_test.npy', y_te); np.save(f'{OUT}/test_index.npy', TE)
done = {(str(r['config']), int(r['seed']), r['model']) for r in res}

def scaled(feats, rows_fit, *row_sets):
    sc = StandardScaler().fit(X.loc[rows_fit, feats].values)
    return [sc.transform(X.loc[r, feats].values) for r in row_sets]

t_all = time.time()
for cfg in CONFIGS:
    feats = FEATURE_SETS[cfg]
    Xtr, Xva, Xte = scaled(feats, TR, TR, VA, TE)
    for seed in SEEDS:
        for name in MODELS:
            if (cfg, seed, name) in done: continue
            t0 = time.time(); m, params, predict = run_model(name, Xtr, y_tr, Xva, y_va, seed); t_fit = time.time() - t0
            t0 = time.time(); p_te = predict(Xte); t_inf = time.time() - t0
            p_va = predict(Xva)
            row = dict(config=cfg, n_features=len(feats), seed=seed, model=name,
                       val_acc=accuracy_score(y_va, p_va), val_macro_f1=mf1(y_va, p_va),
                       test_acc=accuracy_score(y_te, p_te), test_macro_f1=mf1(y_te, p_te),
                       train_s=t_fit, infer_ms_per_1k=1000 * t_inf / len(Xte) * 1000,
                       n_params=n_params(m), size_kb=size_kb(m) if not isinstance(m, torch.nn.Module) else np.nan,
                       params=json.dumps(params, default=str))
            res.append(row); preds[f'{cfg}|{seed}|{name}'] = p_te.astype(np.int8)
            pd.DataFrame(res).to_csv(RES_CSV, index=False); np.savez_compressed(PRED_NPZ, **preds)
            print(f"[{cfg:>8} | seed {seed} | {name:6}] test acc {row['test_acc']*100:6.2f}%  macro-F1 {row['test_macro_f1']:.3f}  ({t_fit:.0f}s)")
print(f'Done in {(time.time() - t_all)/60:.1f} min. Results: {RES_CSV}')

In [ ]:
# Cell 7: summary (mean ± SD over seeds) -> new Table 10/11 source
res = pd.read_csv(f'{OUT}/benchmark_results.csv')
g = res.groupby(['config', 'model'])
summ = pd.DataFrame({
    'n_seeds': g['seed'].nunique(),
    'test_acc_mean': g['test_acc'].mean() * 100, 'test_acc_sd': g['test_acc'].std(ddof=1) * 100,
    'test_f1_mean': g['test_macro_f1'].mean(), 'test_f1_sd': g['test_macro_f1'].std(ddof=1),
    'val_f1_mean': g['val_macro_f1'].mean(),
    'train_s_mean': g['train_s'].mean(), 'infer_ms_per_1k': g['infer_ms_per_1k'].mean(),
}).reset_index()
summ.to_csv(f'{OUT}/summary_mean_sd.csv', index=False)
order = [m for m in MODELS if m in summ['model'].unique()]
fmt = summ.assign(**{'Accuracy (%)': summ.apply(lambda r: f"{r.test_acc_mean:.2f} ± {r.test_acc_sd:.2f}", axis=1),
                     'Macro-F1': summ.apply(lambda r: f"{r.test_f1_mean:.3f} ± {r.test_f1_sd:.3f}", axis=1)})
for metric in ['Accuracy (%)', 'Macro-F1']:
    t = fmt.pivot(index='model', columns='config', values=metric).reindex(order)
    print(f'\n{metric}, test set, mean ± SD over {res.seed.nunique()} seeds'); print(t.to_string())
    t.to_csv(f"{OUT}/table_{'acc' if metric.startswith('Acc') else 'f1'}.csv")
print('\nCost (config 9):'); print(summ[summ.config == '9'].set_index('model').reindex(order)[['train_s_mean', 'infer_ms_per_1k']].round(2).to_string())

In [ ]:
# Cell 8: is FT-Transformer really better? Paired tests on the SAME test records, per seed.
from scipy.stats import binomtest
P = dict(np.load(f'{OUT}/test_predictions.npz')); yt = np.load(f'{OUT}/y_test.npy')
rng = np.random.default_rng(0); B = 2000; n = len(yt)
rows = []
for cfg in CONFIGS:
    for other in ['RF', 'XGB', 'LGBM']:
        for seed in SEEDS:
            a, b = P.get(f'{cfg}|{seed}|FT'), P.get(f'{cfg}|{seed}|{other}')
            if a is None or b is None: continue
            ca, cb = a == yt, b == yt
            n01, n10 = int((ca & ~cb).sum()), int((~ca & cb).sum())   # FT right/other wrong, and the reverse
            p_mc = binomtest(n01, n01 + n10, 0.5).pvalue if n01 + n10 else 1.0
            idx = rng.integers(0, n, (B, n))
            d_acc = ca[idx].mean(1) - cb[idx].mean(1)
            d_f1 = np.array([mf1(yt[i], a[i]) - mf1(yt[i], b[i]) for i in idx[:500]])
            rows.append(dict(config=cfg, comparison=f'FT vs {other}', seed=seed,
                             acc_diff_pts=100 * (ca.mean() - cb.mean()),
                             acc_diff_CI95=f'[{100*np.percentile(d_acc, 2.5):.2f}, {100*np.percentile(d_acc, 97.5):.2f}]',
                             f1_diff=mf1(yt, a) - mf1(yt, b),
                             f1_diff_CI95=f'[{np.percentile(d_f1, 2.5):.3f}, {np.percentile(d_f1, 97.5):.3f}]',
                             FT_only_correct=n01, other_only_correct=n10, mcnemar_p=p_mc))
st = pd.DataFrame(rows); st.to_csv(f'{OUT}/stats_ft_vs_baselines.csv', index=False)
print(st.round(4).to_string(index=False))
agg = st.groupby(['config', 'comparison']).agg(mean_acc_diff=('acc_diff_pts', 'mean'), sd_acc_diff=('acc_diff_pts', 'std'),
                                                 seeds_FT_better=('acc_diff_pts', lambda s: int((s > 0).sum())),
                                                 seeds_p_below_0_05=('mcnemar_p', lambda s: int((s < 0.05).sum())))
print('\nAcross seeds:'); print(agg.round(3).to_string())
print('\nCaveat for the paper: test records are consecutive and not independent, so record-level CIs and McNemar p-values are optimistic.')

In [ ]:
# Cell 9: confusion matrix + per-class report for FT-Transformer and RF (config 9, seed 42)
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, classification_report
P = dict(np.load(f'{OUT}/test_predictions.npz')); yt = np.load(f'{OUT}/y_test.npy')
names = [c.upper() if c in ('dos', 'ddos', 'xss') else c.capitalize() for c in CLASSES]
names = [{'DDOS': 'DDoS', 'DOS': 'DoS', 'XSS': 'XSS'}.get(n, n) for n in names]
for model in ['FT', 'RF']:
    key = f'9|{SEEDS[0]}|{model}'
    if key not in P: continue
    p = P[key]; cm = confusion_matrix(yt, p)
    rep = classification_report(yt, p, target_names=names, digits=4, output_dict=True)
    pd.DataFrame(rep).T.to_csv(f'{OUT}/per_class_{model}_seed{SEEDS[0]}.csv')
    pd.DataFrame(cm, index=names, columns=names).to_csv(f'{OUT}/confusion_{model}_seed{SEEDS[0]}.csv')
    print(f'\n{model}, config 9, seed {SEEDS[0]}'); print(classification_report(yt, p, target_names=names, digits=4))
    fig, ax = plt.subplots(figsize=(6.4, 5.4))
    frac = cm / np.maximum(cm.sum(1, keepdims=True), 1)
    ax.imshow(frac, cmap='Blues', vmin=0, vmax=1)
    for i in range(len(names)):
        for j in range(len(names)):
            ax.text(j, i, cm[i, j], ha='center', va='center', fontsize=9, color='white' if frac[i, j] > 0.5 else 'black')
    ax.set_xticks(range(len(names)), names, rotation=45, ha='right'); ax.set_yticks(range(len(names)), names)
    ax.set_xlabel('Predicted class'); ax.set_ylabel('True class')
    ax.set_title(f'{model}: confusion matrix (test set, seed {SEEDS[0]})', fontsize=11)
    plt.tight_layout(); plt.savefig(f'{OUT}/confusion_{model}_seed{SEEDS[0]}.png', dpi=300); plt.show()

In [ ]:
# Cell 10: capture-identity test (training rows only).
# Normal and DoS were each recorded in separate blocks of rows. Records in two blocks of the SAME class share the
# same behaviour, so a feature that still tells the blocks apart is tracking the capture (host state, time), not the attack.
from sklearn.metrics import roc_auc_score
from scipy.stats import spearmanr, f_oneway

def segments(c):
    idx = np.sort(df.index[df['type'] == c].values)
    return np.split(idx, np.where(np.diff(idx) != 1)[0] + 1)

seg_info = {c: segments(c) for c in CLASSES}
print('Blocks per class:', {c: [f'{s[0]}-{s[-1]} ({len(s)})' for s in v] for c, v in seg_info.items()})
TRset = set(TR)
TR_BY_CLASS = {c: [i for i in np.sort(df.index[df['type'] == c].values) if i in TRset] for c in CLASSES}
multi = [c for c, v in seg_info.items() if len(v) > 1]
TR_SEGS = {c: [s for s in ([i for i in seg if i in TRset] for seg in seg_info[c]) if len(s) >= 10] for c in multi}

def seg_auc(c, f):
    segs = TR_SEGS[c]
    if len(segs) < 2: return np.nan
    vals = np.concatenate([X.loc[s, f].values for s in segs]); lab = np.concatenate([[k] * len(s) for k, s in enumerate(segs)])
    if len(np.unique(vals)) < 2: return 0.5
    aucs = [roc_auc_score(lab == k, vals) for k in range(len(segs))]
    return float(np.mean([max(a, 1 - a) for a in aucs]))

def drift(f):   # mean |Spearman rho| between a feature and record position inside each class (train rows)
    rs = []
    for c in CLASSES:
        v = X.loc[TR_BY_CLASS[c], f].values
        if len(np.unique(v)) > 1: rs.append(abs(spearmanr(np.arange(len(v)), v)[0]))
    return float(np.nanmean(rs)) if rs else np.nan

PAPER_F_REMOVED = ['Process_Pool_Paged Bytes', 'Process_Virtual_Bytes Peak', 'Process_Working Set',
                   'Memory System Driver Total Bytes', 'Process_Pool Nonpaged Bytes',
                   'Memory System Cache Resident Bytes', 'Memory Pool Nonpaged Bytes']   # the 7 removed by the F-audit
def group(f):
    if f in FEATURES_9: return 'kept (paper 9)'
    if f == MCL: return 'Memory Commit Limit'
    if any(norm(p) in norm(f) for p in FINGERPRINT_PATTERNS): return 'removed: fingerprint (hand)'
    if norm(f) in {norm(p) for p in PAPER_F_REMOVED}: return 'removed: F-audit (paper)'
    return 'other'
rows = []
for f in KEEP:
    r = dict(feature=f, group=group(f))
    for c in multi: r[f'{c}_block_AUC'] = seg_auc(c, f)
    r['within_class_drift'] = drift(f)
    r['class_F_train'] = f_oneway(*[X.loc[TR_BY_CLASS[c], f].values for c in CLASSES])[0]
    rows.append(r)
cap = pd.DataFrame(rows)
cap['mean_block_AUC'] = cap[[f'{c}_block_AUC' for c in multi]].mean(1)
cap = cap.sort_values('mean_block_AUC', ascending=False); cap.to_csv(f'{OUT}/capture_identity.csv', index=False)
print('\nMatched as "removed: F-audit (paper)" (expect up to 7; any missing were dropped as near-constant):',
      cap.loc[cap.group == 'removed: F-audit (paper)', 'feature'].tolist())
print('Matched as fingerprint:', cap.loc[cap.group == 'removed: fingerprint (hand)', 'feature'].tolist())
print('\nAUC 0.5 = blocks look the same; AUC 1.0 = the feature alone tells the blocks apart.')
print(cap[cap.group != 'other'].round(3).to_string(index=False))
print('\nBy group (mean):'); print(cap.groupby('group')[['mean_block_AUC', 'within_class_drift']].mean().round(3).to_string())

In [ ]:
# Cell 11: held-out capture block test. Train WITHOUT one block of Normal or DoS, test on that whole block.
# Other classes use their usual training rows. Caps and scaling are refit on each training set.
from collections import Counter
LOSO_MODELS, LOSO_SEEDS = ['RF', 'XGB', 'LGBM', 'FT'], SEEDS[:3]
scen = []
for c in multi:
    segs = seg_info[c]
    for k in range(len(segs)):
        scen.append((c, [s for j, s in enumerate(segs) if j != k], segs[k], f'{c}: train blocks {[j+1 for j in range(len(segs)) if j != k]}, test block {k+1}'))
rows = []
for c, train_segs, test_seg, label in scen:
    own = np.sort(np.concatenate(train_segs)); n_own_va = max(1, int(0.15 * len(own)))   # last 15% of own blocks = validation
    tr_rows = list(TR[y.loc[TR].values != CLASSES.index(c)]) + list(own[:-n_own_va])
    va_rows = list(VA[y.loc[VA].values != CLASSES.index(c)]) + list(own[-n_own_va:])
    te_rows = list(test_seg)
    for cfg in ['9', '10']:
        feats = FEATURE_SETS[cfg]
        raw_f = df[feats]
        caps = raw_f.loc[tr_rows].quantile(CAP_Q); Z = raw_f.clip(upper=caps, axis=1)
        sc = StandardScaler().fit(Z.loc[tr_rows].values)
        Ztr, Zva, Zte = (sc.transform(Z.loc[r].values) for r in (tr_rows, va_rows, te_rows))
        for seed in LOSO_SEEDS:
            for name in LOSO_MODELS:
                m, params, predict = run_model(name, Ztr, y.loc[tr_rows].values, Zva, y.loc[va_rows].values, seed)
                p = predict(Zte); target = CLASSES.index(c)
                wrong = Counter(CLASSES[i] for i in p if i != target).most_common(1)
                rows.append(dict(scenario=label, block_size=len(te_rows), config=cfg, seed=seed, model=name,
                                 recall_on_heldout_block=float((p == target).mean()),
                                 most_common_wrong=wrong[0][0] if wrong else ''))
                print(f'{label} | {cfg} | seed {seed} | {name}: recall {rows[-1]["recall_on_heldout_block"]:.3f}')
lo = pd.DataFrame(rows); lo.to_csv(f'{OUT}/heldout_block_results.csv', index=False)
P = dict(np.load(f'{OUT}/test_predictions.npz')); yt = np.load(f'{OUT}/y_test.npy')
within = {(c, m): np.mean([(P[f'9|{s}|{m}'][yt == CLASSES.index(c)] == CLASSES.index(c)).mean() for s in SEEDS if f'9|{s}|{m}' in P])
          for c in multi for m in LOSO_MODELS}
print('\nWithin-session test recall (main benchmark, config 9) for comparison:', {f'{c}/{m}': round(v, 3) for (c, m), v in within.items()})
print(lo.groupby(['scenario', 'config', 'model'])['recall_on_heldout_block'].agg(['mean', 'std']).round(3).to_string())

In [ ]:
# Cell 12: Windows 7 compatibility check. Upload Train_Test_Windows_7.csv first (TON_IoT, same folder as the Windows 10 file).
w7 = glob.glob('/content/**/Train_Test_Windows_7*.csv', recursive=True)
if not w7:
    print('No Windows 7 file uploaded; skipping. (The paper used only the Windows 10 file.)')
else:
    d7 = pd.read_csv(w7[0], low_memory=False)
    print('Windows 7 shape:', d7.shape); print(d7['type'].value_counts() if 'type' in d7 else 'no "type" column')
    n7 = {norm(c): c for c in d7.columns}
    for f in FEATURES_9 + [MCL]:
        print(f"{'FOUND  ' if norm(f) in n7 else 'MISSING'} {f}" + (f"  ->  {n7[norm(f)]}" if norm(f) in n7 else ''))
    print('\nIf most of the 9 are MISSING, a direct cross-dataset test is not possible; report that as a limitation.')

In [ ]:
# Cell 13: zip everything and download it. Send this zip to Claude / the professors.
import shutil
z = shutil.make_archive(OUT.rstrip('/'), 'zip', OUT)
print('Created', z)
try:
    from google.colab import files; files.download(z)
except Exception as e:
    print('Download manually from the Files panel:', e)